<a href="https://colab.research.google.com/github/opherdonchin/BayesShortCourse/blob/main/sleep/solved/05_varying_intercept_slope.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sleep deprivation 5 — Varying intercepts and slopes

Allow participants to differ both at baseline and in their response to accumulated sleep deprivation.

## Setup

This course pins PyMC, modular ArviZ, and Bambi for reproducibility because their APIs can change across major versions.

In [ ]:
%pip install -q \
    "pandas==2.2.3" \
    "pymc==6.3.2" \
    "arviz-base==1.3.0" \
    "arviz-stats==1.3.2" \
    "arviz-plots[matplotlib]==1.3.1" \
    "bambi==0.21.0"

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import bambi as bmb
import pymc as pm
import arviz_base as azb
import arviz_stats as azs
import arviz_plots as azp

RANDOM_SEED = 20260924
azp.style.use("arviz-variat")

print("PyMC:", pm.__version__)
print("Bambi:", bmb.__version__)
print("arviz-base:", azb.__version__)
print("arviz-plots:", azp.__version__)
print("arviz-stats:", azs.__version__)

## Data

The original study contains two adaptation/training days followed by a baseline measurement and then seven nights of severe sleep restriction. Following the chapter, we drop original days 0–1 and subtract 2 from the remaining day number. Therefore **`Days = 0` is the baseline measurement before sleep deprivation begins**.

That zero point is scientifically meaningful, so every Bambi model in this sequence uses `center_predictors=False`. The `Intercept` prior is therefore a prior on baseline reaction time rather than reaction time at the average deprivation day.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/lme4/sleepstudy.csv"

sleep = pd.read_csv(DATA_URL).drop(columns="rownames")
sleep = sleep.loc[sleep["Days"] >= 2].copy()
sleep["Days"] = sleep["Days"] - 2
sleep["Subject"] = sleep["Subject"].astype(str)
sleep = sleep.reset_index(drop=True)

print(f"{sleep['Subject'].nunique()} participants, {len(sleep)} observations")
print(f"Days: {sleep['Days'].min()} to {sleep['Days'].max()}")
sleep.head()

# 5.1 Participant-specific deprivation effects

Do participants differ in how strongly continued sleep deprivation affects reaction time?

$$\mu_i=(\alpha+u_{0,s[i]})+(\beta+u_{1,s[i]})\,\text{Days}_i.$$

**Bambi note.** The book's `brms` model places an LKJ prior on the correlation between participant intercept and slope deviations. Bambi currently represents these group-specific terms with independent hierarchical priors, so this notebook reproduces the varying-intercept/varying-slope structure but not that correlation parameter.

In [ ]:
priors = {
    "Intercept": bmb.Prior("Normal", mu=250, sigma=100),
    "Days": bmb.Prior("Normal", mu=0, sigma=20),
    "sigma": bmb.Prior("Exponential", lam=0.04),
    "1|Subject": bmb.Prior(
        "Normal", mu=0, sigma=bmb.Prior("Exponential", lam=0.04)
    ),
    "Days|Subject": bmb.Prior(
        "Normal", mu=0, sigma=bmb.Prior("Exponential", lam=0.10)
    ),
}
model = bmb.Model(
    "Reaction ~ Days + (1 + Days | Subject)",
    sleep, family="gaussian", priors=priors, categorical="Subject",
    center_predictors=False,
)
model

In [ ]:
prior = model.prior_predictive(draws=500, random_seed=RANDOM_SEED)
azp.plot_ppc_dist(
    prior,
    group="prior_predictive",
    var_names=["Reaction"],
    kind="ecdf",
    figure_kwargs={"figsize": (7, 4)},
);

# 5.2 Individual trajectories

What population and individual trajectories are estimated when both baseline and slope partially pool?

In [ ]:
idata = model.fit(draws=1000, tune=1500, chains=4, target_accept=0.95, random_seed=RANDOM_SEED)
print("Divergences:", int(idata["sample_stats"]["diverging"].sum().item()))


In [ ]:
azs.summary(
    idata, var_names=["Intercept", "Days", "sigma", "1|Subject_sigma", "Days|Subject_sigma"],
    ci_prob=0.90, ci_kind="hdi", round_to=2,
)

In [ ]:
azp.plot_trace_dist(idata, var_names=["Intercept", "Days", "sigma", "1|Subject_sigma", "Days|Subject_sigma"]);

In [ ]:
bmb.interpret.plot_predictions(
    model,
    idata,
    conditional="Days",
    average_by="Subject",
    target="mean",
    prob=[0.50, 0.90],
)

In [ ]:
bmb.interpret.plot_predictions(
    model,
    idata,
    conditional=["Days", "Subject"],
    target="mean",
    prob=0.90,
    subplot_kwargs={"main": "Days", "panel": "Subject"},
    fig_kwargs={"wrap": 6},
)

# 5.3 Remaining predictive mismatch

After allowing both intercepts and slopes to vary, what predictive mismatch remains?

In [ ]:
model.predict(
    idata,
    kind="response",
    inplace=True,
    random_seed=RANDOM_SEED,
)

azp.plot_ppc_dist(
    idata,
    var_names=["Reaction"],
    kind="ecdf",
    figure_kwargs={"figsize": (7, 4)},
);

The hierarchical mean structure is now flexible, but reaction times are positive and right-skewed. The next branch asks whether a positive-only lognormal observation model is a useful alternative.